[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camenduru/comfyui-colab/blob/main/stable/stable_diffusion_1_5_comfyui_colab.ipynb)

In [ ]:
import os

# ── 1. 系統套件 ──────────────────────────────────────────
!apt -y update -qq
!apt -y install -qq aria2 ffmpeg libgl1

# ── 2. PyTorch（Colab 預裝通常已夠，但確保 CUDA 對齊）──
# Colab 目前是 CUDA 12.x，直接用預裝的 torch 即可
# 若遇到版本問題再取消下面這行的註解
# !pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121 -U

# ── 3. Clone 最新版 ComfyUI ────────────────────────────
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# ── 4. Clone SeedVR2 custom node ─────────────────────
!git clone https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler \
    /content/ComfyUI/custom_nodes/seedvr2_videoupscaler

# ── 5. 安裝 SeedVR2 依賴 ─────────────────────────────
!pip install -q -r /content/ComfyUI/custom_nodes/seedvr2_videoupscaler/requirements.txt

# ── 6. 選裝加速套件（Colab T4 不支援 flash-attn，A100/L4 才有效）
import subprocess, sys
gpu_info = subprocess.check_output("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True).decode().strip()
print(f"GPU: {gpu_info}")

if any(x in gpu_info for x in ["A100", "L4", "H100"]):
    print("安裝 flash-attn（支援 Ampere+ 架構）...")
    !pip install -q flash-attn --no-build-isolation
else:
    print("T4 不支援 flash-attn，跳過（用 PyTorch SDPA 替代，效能稍慢但可用）")

print("\n✅ 環境安裝完成")

In [ ]:
import os
os.makedirs("/content/ComfyUI/models/SEEDVR2", exist_ok=True)

# ── 選擇模型版本 ──────────────────────────────────────
# Colab T4  (15GB VRAM) → 建議用 3B FP16，並開 BlockSwap
# Colab A100 (40/80GB)  → 可用 7B FP16 或 3B FP16

MODEL = "3B_FP16"   # 改成 "7B_FP16" 或 "3B_FP8" 視 VRAM 而定

MODEL_URLS = {
    "3B_FP16": (
        "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/seedvr2_ema_3b_fp16.safetensors",
        "seedvr2_ema_3b_fp16.safetensors"
    ),
    "3B_FP8": (
        "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/seedvr2_ema_3b_fp8.safetensors",
        "seedvr2_ema_3b_fp8.safetensors"
    ),
    "7B_FP16": (
        "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/seedvr2_ema_7b_fp16.safetensors",
        "seedvr2_ema_7b_fp16.safetensors"
    ),
}

VAE_URL = "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/ema_vae_fp16.safetensors"

model_url, model_filename = MODEL_URLS[MODEL]
model_path = f"/content/ComfyUI/models/SEEDVR2/{model_filename}"
vae_path = "/content/ComfyUI/models/SEEDVR2/ema_vae_fp16.safetensors"

# 下載 DiT 模型
if not os.path.exists(model_path):
    print(f"下載 {MODEL} DiT 模型（可能需要 5-15 分鐘）...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
        "{model_url}" \
        -d /content/ComfyUI/models/SEEDVR2 \
        -o "{model_filename}"
else:
    print(f"✅ DiT 模型已存在，跳過下載")

# 下載 VAE 模型
if not os.path.exists(vae_path):
    print("下載 VAE 模型...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
        "{VAE_URL}" \
        -d /content/ComfyUI/models/SEEDVR2 \
        -o "ema_vae_fp16.safetensors"
else:
    print(f"✅ VAE 模型已存在，跳過下載")

# 確認
!ls -lh /content/ComfyUI/models/SEEDVR2/

In [ ]:
import atexit, requests, subprocess, time, re, os, threading, socket
from random import randint
from queue import Queue
from datetime import datetime

# ── 工具函式 ─────────────────────────────────────────────
def find_free_port(start=8188, end=8999):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            if s.connect_ex(("127.0.0.1", port)) != 0:
                return port
    raise RuntimeError("找不到可用連接埠")

def safe_run(cmd):
    return subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ── 0) 安全更新 cloudflared（避免 Text file busy）────────────
# 不直接覆寫執行中的 /content/cloudflared，而是下載到新檔後原子替換。
safe_run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/content/cloudflared.new"])
safe_run(["chmod", "+x", "/content/cloudflared.new"])
safe_run(["mv", "-f", "/content/cloudflared.new", "/content/cloudflared"])

# ── 1) 若前一次殘留 cloudflared，先清掉（避免 tunnel 互撞）────
safe_run(["pkill", "-f", "/content/cloudflared tunnel --url"])

# ── 2) 動態選擇可用埠與 DB 檔（避免 address in use / DB lock）──
comfy_port = find_free_port(8188, 8999)
metrics_port = find_free_port(9000, 9800)
db_path = f"/content/ComfyUI/user/comfyui_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"

# ── 3) Cloudflare Tunnel ───────────────────────────────
def cloudflared(port, metrics_port, output_queue):
    proc = subprocess.Popen(
        [
            "/content/cloudflared", "tunnel",
            "--url", f"http://127.0.0.1:{port}",
            "--metrics", f"127.0.0.1:{metrics_port}",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    atexit.register(lambda p: p.terminate(), proc)

    tunnel_url = None
    for _ in range(30):
        time.sleep(2)
        try:
            m = re.search(
                r"(?P<url>https?://[^\s]+\.trycloudflare\.com)",
                requests.get(f"http://127.0.0.1:{metrics_port}/metrics", timeout=2).text,
            )
            if m:
                tunnel_url = m.group("url")
                break
        except Exception:
            pass

    output_queue.put(tunnel_url)

# 先啟動 Cloudflare 執行緒（背景抓網址）
output_queue = Queue()
threading.Thread(target=cloudflared, args=(comfy_port, metrics_port, output_queue), daemon=True).start()

print("⏳ 正在啟動 Cloudflare 與 ComfyUI...")
print(f"🔌 ComfyUI Port: {comfy_port}")

# ── 4) 啟動 ComfyUI（前台輸出 log）──────────────────────
def run_comfyui():
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        [
            "python", "/content/ComfyUI/main.py",
            "--dont-print-server",
            "--listen", "0.0.0.0",
            "--port", str(comfy_port),
            "--database-url", f"sqlite:///{db_path}",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    def print_url():
        url = output_queue.get()
        print("\n" + "=" * 60)
        print("✅ ComfyUI 已就緒！")
        if url:
            print(f"🌐 存取連結：{url}")
        else:
            print("⚠️ Cloudflare URL 取得失敗，請檢查 cloudflared 是否啟動成功")
        print("=" * 60 + "\n")

    threading.Thread(target=print_url, daemon=True).start()

    try:
        for line in process.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        process.terminate()

run_comfyui()